### DATA LOADING AND PREPROCESSING

In [1]:
import os
import pandas as pd
import numpy as np
from collections import Counter

## Merge Strategy

Before feature extraction, we needed to combine the accelerometer and gyroscope signals for each recording.

Sensors used: Accelerometer (ax, ay, az) and Gyroscope (gx, gy, gz)

Issue: Row counts differed slightly (450–600 rows per recording) due to independent sensor timing and OS scheduling, even though both were recorded at 50 Hz.

Solution: We merged the sensors using nearest-neighbor timestamp matching based on the seconds_elapsed column.

Result: A clean, aligned time-series per recording with the following format:

| seconds_elapsed | ax | ay | az | gx | gy | gz | activity

This ensures temporal consistency and preserves sequential integrity for HMM modeling.

In [2]:
DATA_PATH = "data"

In [3]:
all_samples = []

In [4]:
for participant in os.listdir(DATA_PATH):
    participant_path = os.path.join(DATA_PATH, participant)
    
    if not os.path.isdir(participant_path):
        continue
    
    for recording in os.listdir(participant_path):
        recording_path = os.path.join(participant_path, recording)
        
        accel_path = os.path.join(recording_path, "Accelerometer.csv")
        gyro_path = os.path.join(recording_path, "Gyroscope.csv")
        
        if not os.path.exists(accel_path) or not os.path.exists(gyro_path):
            continue
        
        # Load data
        accel = pd.read_csv(accel_path)
        gyro = pd.read_csv(gyro_path)
        
        # Keep necessary columns
        accel = accel[["seconds_elapsed", "x", "y", "z"]]
        gyro = gyro[["seconds_elapsed", "x", "y", "z"]]
        
        # Rename columns
        accel.rename(columns={"x":"ax", "y":"ay", "z":"az"}, inplace=True)
        gyro.rename(columns={"x":"gx", "y":"gy", "z":"gz"}, inplace=True)
        
        # Sort by time
        accel.sort_values("seconds_elapsed", inplace=True)
        gyro.sort_values("seconds_elapsed", inplace=True)
        
        # Merge using nearest timestamp
        merged = pd.merge_asof(
            accel,
            gyro,
            on="seconds_elapsed",
            direction="nearest"
        )
        
        # Compute duration
        duration = merged["seconds_elapsed"].max() - merged["seconds_elapsed"].min()
        
        # Extract activity label from folder name
        activity = recording.split("-")[0]
        
        merged["activity"] = activity
        
        all_samples.append({
            "data": merged,
            "activity": activity,
            "participant": participant,
            "duration": duration
        })

print(f"Total recordings loaded: {len(all_samples)}")

Total recordings loaded: 50


In [5]:
durations = [sample["duration"] for sample in all_samples]
print("Min duration:", min(durations))
print("Max duration:", max(durations))

Min duration: 7.720366943359376
Max duration: 11.813686767578124


In [6]:
activities = [sample["activity"] for sample in all_samples]
print(Counter(activities))

Counter({'Jump': 13, 'Walk': 13, 'Stand': 12, 'Still': 12})


In [7]:
merged.head()

,seconds_elapsed,ax,ay,az,gx,gy,gz,activity
0,0.294608,0.252918,0.768914,0.447187,-0.135438,0.404938,-0.195112,Walk
1,0.314664,0.165534,0.655931,0.511350,-0.234437,0.300575,-0.217525,Walk
2,0.334720,0.052428,0.354345,0.781880,-0.264275,0.334675,-0.234575,Walk
3,0.354776,0.057142,0.384276,0.931904,-0.232375,0.481663,-0.241038,Walk
4,0.374832,-0.147886,0.246621,0.606323,-0.119350,0.482762,-0.278300,Walk


## Windowing Strategy

To convert each continuous recording into fixed-length sequences suitable for feature extraction and HMM training:

Window Size: 2 seconds (100 samples at 50 Hz)

Overlap: 50% (50 samples shift between windows)

Reasoning:

Captures full cycles of walking (~1–2 Hz)

Stabilizes variance for standing and still activities

Retains resolution for frequency-domain analysis such as FFT

Overlapping windows increase the number of training sequences

Results:

Each 10-second recording produced ~9 windows

With 50 recordings, the dataset contains approximately 450 feature windows ready for feature extraction

In [8]:
def window_data(df, window_size=100, overlap=50):
    """
    df: merged dataframe with columns ['seconds_elapsed','ax','ay','az','gx','gy','gz','activity','participant']
    window_size: number of samples per window
    overlap: number of samples to overlap
    """
    windows = []
    start = 0
    end = window_size
    
    while end <= len(df):
        window_df = df.iloc[start:end].copy()
        
        # Store window data and labels
        windows.append({
            "ax": window_df["ax"].values,
            "ay": window_df["ay"].values,
            "az": window_df["az"].values,
            "gx": window_df["gx"].values,
            "gy": window_df["gy"].values,
            "gz": window_df["gz"].values,
            "activity": window_df["activity"].iloc[0]
        })
        
        start += (window_size - overlap)
        end = start + window_size
    
    return windows

In [9]:
all_windows = []

In [10]:
for sample in all_samples:
    merged_df = sample["data"]
    windows = window_data(merged_df, window_size=100, overlap=50)
    all_windows.extend(windows)

print(f"Total windows created: {len(all_windows)}")

Total windows created: 398


In [11]:
activities = [w["activity"] for w in all_windows]
print(Counter(activities))

Counter({'Walk': 104, 'Jump': 100, 'Stand': 99, 'Still': 95})
